# McKenzie et al. (2025) on dandiset 000552: settling of the CA1 code after transfer

Paper: McKenzie, Sommer, Donaldson, Pimentel, Kakani, Choi, Newman & English (2025), "Event boundaries
drive norepinephrine release and distinctive neural representations of space in the rodent
hippocampus", eLife 14:RP105183, [doi:10.7554/elife.105183](https://doi.org/10.7554/elife.105183).
The claims file (`claims/mckenzie-2025.yaml`) was written from the eLife Reviewed Preprint version 1
(January 31, 2025), with the legends of Figures S1 to S7.

Dataset: [DANDI:000552](https://dandiarchive.org/dandiset/000552/0.230630.2304) version
**0.230630.2304** (Huszar et al. 2022). This is the version the paper cites (Methods, Electrophysiology
subjects: "The complete dataset is available at https://dandiarchive.org/dandiset/000552/0.230630.2304"),
so it is pinned, as the protocol asks. A later version, 0.261113.1035 (2026-11-13), holds the same spike
times (checked below, unit by unit) and adds CellExplorer cell types and epoch names; only those labels
are read from it, because the pinned version has none (claims file E3). Every problem found in the pinned
version is also looked for in the later one.

This notebook tests the claims on population vectors, nearest neighbours, decorrelation and place
fields (Figures 7, S6, S7). The reactivation claims (Figure 8) are in `reactivation.ipynb`. The
photometry claims (Figures 1 to 6, S1 to S5) are blocked: those recordings are not in the dandiset.

| Claim | Here |
|---|---|
| figs1a-desipramine, figs1b-yohimbine, fig1d-arena-decay, figs1e-univariate-covariates, fig1e-arena-time-from-transition, fig2c-track-time-from-transition, figs2-no-reward-modulation, fig3d-object-introduction, figs3-object-order-and-sampling, fig4b-novel-object-on-track, fig5d-decay-hastens-with-days, figs4-behavior-day1-day2, fig6-recent-history | blocked, data missing (photometry) |
| ne-and-settling-time-course | blocked, data missing (the NE half) |
| ephys-ensemble-size | tested |
| figs6b-decorrelation-novel, figs6d-decorrelation-familiar | tested |
| fig7ab-umap-embedding | tested |
| fig7e-novel-settling, fig7f-familiar-settling, fig7g-familiar-more-similar-early | tested |
| figs7ab-in-field-out-field, figs7cd-movement-not-time-locked, figs7ef-time-explains-similarity, figs7gh-unconditioned-similarity | tested |
| fig8b-late-moments-reactivated, fig8d-familiar-no-difference, fig8-no-pre-reactivation | tested in `reactivation.ipynb` |

In [ ]:
import time, zlib, warnings, resource
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import settle as S

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200, "display.max_columns", 40)
T_START = time.time()

def seed(claim, k=0):
    return zlib.crc32(f"mckenzie-2025/{claim}".encode()) + k

print("dandiset", S.DANDISET_ID, "version", S.VERSION, "labels from", S.LABEL_VERSION)

## Data check

Each session file holds the units (spike times only, in the pinned version), the head position
(x, y and a linearized position), trials, the ripples detected by Huszar et al., sleep states and,
in most files, unlabeled epochs. The sessions of the three mice the Methods name are loaded. The maze
runs are found as stretches of tracking; time 0 of a run (transfer, claims file E2) is the start of
the epoch that holds it.

In [ ]:
SESS = {}
LABELS = {}
for sid in S.SESSIONS:
    s = S.load_session(sid)
    ct, env = S.cell_types(sid, s["spike_index"])   # raises if the units differ between versions
    s["cell_type"] = ct
    SESS[sid] = s
    LABELS[sid] = env
print(f"loaded {len(SESS)} sessions in {time.time() - T_START:.0f} s")

In [ ]:
def env_of(sid, t0, t1):
    for a, b, e in LABELS[sid]:
        if a <= (t0 + t1) / 2 < b:
            return e
    return "?"

rows = []
for sid, s in SESS.items():
    ct = s["cell_type"]
    for k, r in enumerate(s["runs"]):
        i = r["idx"]
        rows.append(dict(session=sid, units=len(s["spikes"]), pyr=int((ct == "Pyramidal Cell").sum()),
                         narrow=int((ct == "Narrow Interneuron").sum()), wide=int((ct == "Wide Interneuron").sum()),
                         run=k, t0=round(r["t0"], 1), dur_s=round(r["t1"] - r["t0"]), tracked_s=round(r["last_track"] - r["first_track"]),
                         first_track_s=round(r["first_track"] - r["t0"], 1), trials=r["n_trials"], familiar=r["familiar"],
                         env_label_version=env_of(sid, r["first_track"], r["last_track"]),
                         x_range=f"{np.nanmin(s['xy'][i, 0]):.2f}..{np.nanmax(s['xy'][i, 0]):.2f}",
                         y_range=f"{np.nanmin(s['xy'][i, 1]):.2f}..{np.nanmax(s['xy'][i, 1]):.2f}",
                         lin_cm=f"{np.nanmin(s['lin'][i]):.0f}..{np.nanmax(s['lin'][i]):.0f}",
                         frame_s=round(float(np.median(np.diff(s['pos_t'][i]))), 4)))
runs_df = pd.DataFrame(rows)
print(runs_df.to_string())
print()
for sid, s in SESS.items():
    st, lab = s["sleep"]
    print(sid, "epochs:", None if s["epochs"] is None else len(s["epochs"]),
          "| ripples:", len(s["rip"]), f"duration {s['rip_dur'].min()*1e3:.0f} to {s['rip_dur'].max()*1e3:.0f} ms",
          "| sleep labels:", sorted(set(lab)), "| position unit:", s["xy_unit"],
          "| spikes sorted:", all(np.all(np.diff(x) >= 0) for x in s["spikes"]))

What the check shows, against the paper and the files' own description:

- The spike times of every unit are identical in the pinned and the later version (`cell_types` compares
  the spike index of every file, and the spike times were compared at both ends while preparing).
- Ripple durations lie within 20 to 200 ms in every session, the Methods' limits (Ripple detection).
- Position is sampled at 30 Hz (25 Hz in three novel linear-track runs). The x, y position is stated in
  cm; it spans 0 to about 92 in every familiar run, but 0 to 1 in five of the nine novel runs (below).
  The linearized position is in cm in all runs, so it is the position used here.
- Nine sessions have a familiar figure-eight run followed by a tracked novel run. Three more
  (e13-26m1-211124, e15-13f1-220117, e16-3m2-211210) have one tracked run only; the later version's
  labels say their second maze epoch is an open field, which has no tracking in the pinned version.

In [ ]:
f_old = S.open_file("e16-3m2-211211"); f_new = S.open_file("e16-3m2-211211", S.LABEL_VERSION)
print("pinned", S.VERSION, "units columns:", sorted(f_old["units"].keys()))
print("later ", S.LABEL_VERSION, "units columns:", sorted(f_new["units"].keys()))
print("pinned epochs columns:", sorted(f_old["intervals/epochs"].keys()))
print("later  epochs columns:", sorted(f_new["intervals/epochs"].keys()))
print("e13-26m1-211116 pinned intervals:", list(S.open_file("e13-26m1-211116")["intervals"].keys()))

In [ ]:
for sid, s in SESS.items():
    for r in s["runs"]:
        i = r["idx"]; xy = s["xy"][i]
        if np.nanmax(xy) <= 1.5:
            print(f"{sid} run at {r['t0']:.0f} s: x {np.nanmin(xy[:,0]):.2f}..{np.nanmax(xy[:,0]):.2f}, "
                  f"y {np.nanmin(xy[:,1]):.2f}..{np.nanmax(xy[:,1]):.2f}, unit '{s['xy_unit']}', linearized {np.nanmin(s['lin'][i]):.0f}..{np.nanmax(s['lin'][i]):.0f} cm")
f = S.open_file("e13-26m1-211019", S.LABEL_VERSION)
g = f["processing/behavior/SubjectPosition/SpatialSeries2_tmaze_ipshita/data"]
print("later version, e13-26m1-211019 novel run:", g.attrs["unit"], np.nanmin(g[:, 0]).round(2), np.nanmax(g[:, 0]).round(2))

**Clock check.** Position and spikes must share a clock. For every run, the mean spatial information of
the pyramidal units is computed with the position as recorded and shifted by 5, 30 and 120 s; on a
shared clock it peaks at zero shift. This is the check the protocol asks for with continuous data.

In [ ]:
clock = []
for sid, s in SESS.items():
    pyr = np.where(s["cell_type"] == "Pyramidal Cell")[0]
    for r in s["runs"]:
        V = S.run_vectors(s, r, pyr); B = S.run_behaviour(s, r, V["edges"])
        si = S.spatial_info_by_shift(V["counts"], B["lin"])
        clock.append(dict(session=sid, t0=round(r["t0"]), env=env_of(sid, r["first_track"], r["last_track"]),
                          **{f"shift {k:+d} s": round(v, 3) for k, v in si.items()}))
clock = pd.DataFrame(clock)
clock["peak_at_0"] = clock[[c for c in clock if c.startswith("shift")]].idxmax(1) == "shift +0 s"
print(clock.to_string())

In [ ]:
# e16-3m2-211211: the novel-run tracking and its trials sit in the epoch the later version names an
# open field (9559 to 12002 s); the later version names the next maze epoch (15124 to 16771 s) the linear track.
sid = "e16-3m2-211211"; s = SESS[sid]; r = s["runs"][1]
pyr = np.where(s["cell_type"] == "Pyramidal Cell")[0]
print("epochs:", [(round(a), round(b), e) for a, b, e in LABELS[sid]])
print(f"tracking as filed: {s['pos_t'][r['idx']][0]:.1f} to {s['pos_t'][r['idx']][-1]:.1f} s, linearized "
      f"{np.nanmin(s['lin'][r['idx']]):.0f}..{np.nanmax(s['lin'][r['idx']]):.0f} cm; trials {r['n_trials']}")
V = S.run_vectors(s, r, pyr); B = S.run_behaviour(s, r, V["edges"])
print("spatial information, tracking as filed:", {k: round(v, 3) for k, v in S.spatial_info_by_shift(V["counts"], B["lin"], (-30, -5, 0, 5, 30)).items()})
lin_ep = [(a, b) for a, b, e in LABELS[sid] if e == "linear_manu"][0]
off = lin_ep[0] - r["t0"]
s2 = dict(s); s2["pos_t"] = s["pos_t"].copy(); s2["pos_t"][r["idx"]] += off
r2 = dict(r, t0=lin_ep[0], t1=lin_ep[1])
V2 = S.run_vectors(s2, r2, pyr); B2 = S.run_behaviour(s2, r2, V2["edges"])
print(f"spatial information, tracking moved by {off:.1f} s onto the linear_manu epoch:",
      {k: round(v, 3) for k, v in S.spatial_info_by_shift(V2["counts"], B2["lin"], (-30, -5, 0, 5, 30)).items()})
f = S.open_file(sid, S.LABEL_VERSION)
ts = f["processing/behavior/SubjectPosition/SpatialSeries3_linear_manu/timestamps"]
print(f"later version: linear_manu tracking {ts[0]:.1f} to {ts[-1]:.1f} s, inside its open-field epoch too")

In [ ]:
for sid, s in SESS.items():
    print(sid, "| behavior module description:", repr(s["behavior_description"]))

## Sessions (claims file E1)

The sessions that fit the Methods' description are those with a familiar figure-eight run and a tracked
novel run: nine. In e13-26m1-211116 the novel epoch is missing from the pinned file and the tracking
lasts 344 s, so it has no minute 10; the tests use the other **eight**, the paper's N. By the later
version's labels these are 3 novel figure-eight mazes (`tmaze_ipshita`), 4 linear tracks and, in
e16-3m2-211211, an open field whose time holds linear-track tracking (above); the paper lists 3 linear
tracks, 3 figure-eight mazes and 1 arena. Where a session had two novel environments, the pinned version
tracks only one, so E1's readings (a) and (b) are the same here.

e16-3m2-211211's novel run is the open-field epoch; analyses that do not use position keep it. Analyses
conditioned on location use its position as filed in the main reading (the paper's eight sessions) and
are also shown on the seven sessions whose tracking passes the clock check.

In [ ]:
ALL9 = [sid for sid, s in SESS.items() if len(s["runs"]) >= 2]
TEST = [sid for sid in ALL9 if SESS[sid]["runs"][1]["t1"] - SESS[sid]["runs"][1]["t0"] >= 660]
CLOCK_OK = [sid for sid in TEST if sid != "e16-3m2-211211"]
print("sessions with familiar and novel runs:", len(ALL9), ALL9)
print("tested (novel run at least 660 s):", len(TEST))
print("with tracking that passes the clock check:", len(CLOCK_OK))
for sid in TEST:
    s = SESS[sid]; f, n = s["runs"][0], s["runs"][1]
    print(f"  {sid}: familiar {f['t0']:.0f}+{f['t1']-f['t0']:.0f} s ({env_of(sid, f['first_track'], f['last_track'])}), "
          f"novel {n['t0']:.0f}+{n['t1']-n['t0']:.0f} s ({env_of(sid, n['first_track'], n['last_track'])}), familiar flag of trials {f['familiar']}/{n['familiar']}")

In [ ]:
t0 = time.time()
MEAS = {}      # (session, 'fam'|'nov', 'pyr'|'all') -> measures
for sid in TEST:
    s = SESS[sid]
    pyr = np.where(s["cell_type"] == "Pyramidal Cell")[0]
    for env, r in (("fam", s["runs"][0]), ("nov", s["runs"][1])):
        MEAS[(sid, env, "pyr")] = S.run_measures(s, r, pyr)
        MEAS[(sid, env, "all")] = S.run_measures(s, r, np.arange(len(s["spikes"])))
print(f"measures for {len(MEAS)} runs in {time.time() - t0:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e6:.1f} GB")
print("units dropped (no spike in the first 1000 s):", {f"{k[0]} {k[1]} {k[2]}": v["dropped"] for k, v in MEAS.items() if v["dropped"]})

## ephys-ensemble-size

> The analyzed CA1 ensembles contain on average 253.8 neurons per session, ranging from 191 to 363.

Criterion: over the sessions of E1, (a) the pyramidal units or (b) all units; reproduced when, under one
reading, the mean is within 25.4 of 253.8 and every session lies within 172 to 399.

In [ ]:
cnt = pd.DataFrame([dict(session=sid, all_units=len(SESS[sid]["spikes"]),
                         pyramidal=int((SESS[sid]["cell_type"] == "Pyramidal Cell").sum())) for sid in ALL9])
print(cnt.to_string())
for name, sel in (("8 tested sessions", TEST), ("9 sessions with a novel run", ALL9)):
    c = cnt.set_index("session").loc[sel]
    for col in ("pyramidal", "all_units"):
        v = c[col].values
        ok = abs(v.mean() - 253.8) <= 25.4 and v.min() >= 172 and v.max() <= 399
        print(f"{name}, {col}: mean {v.mean():.2f}, range {v.min()} to {v.max()} -> {'within' if ok else 'outside'} tolerance")

**Result: reproduced under reading (b), all units.** Over the eight tested sessions all units give a mean of
259.38 (191 to 363); over the nine sessions with a novel run the mean is 253.78 and the range 191 to
363, the paper's numbers. Pyramidal units alone give a mean of 215.75 over the eight sessions, outside the
tolerance. The paper's ensemble count thus matches all units of nine sessions, while its Methods analyze
pyramidal cells only and its tests have 8 sessions.

## figs6b-decorrelation-novel and figs6d-decorrelation-familiar

> In a novel environment, CA1 population vectors decorrelate faster with lag in the first minute after transfer than at minute 10.
> In the familiar environment the decorrelation rate does not differ between the first minute and minute 10 after transfer.

Per session and window, the mean Pearson R of z-scored pyramidal vectors at lags 0.1 to 10 s (both bins
in the window), fit with R = beta exp(-lag/tau) + c; the rate is 1/tau. Window readings: (a) the 60 s of
the minute, (b) 100 s from its start; minute 10 as 600 or 540 s. Paired t-test over the eight sessions.

In [ ]:
def decay_table(env, units="pyr", sessions=TEST):
    rows = []
    for sid in sessions:
        M = MEAS[(sid, env, units)]
        row = dict(session=sid)
        for wname, length in (("a60", 60.0), ("b100", 100.0)):
            for mname, start in (("min1", 0.0), ("min10", 600.0), ("min10alt", 540.0)):
                lag, r = S.decorrelation_curve(M["zn"], M["t"], start, length)
                row[f"{wname}_{mname}"] = S.decay_rate(lag, r)[0]
                row[f"{wname}_{mname}_r0.1"] = r[0]
        rows.append(row)
    return pd.DataFrame(rows).set_index("session")

def paired(a, b):
    ok = np.isfinite(a) & np.isfinite(b)
    t, p = stats.ttest_rel(a[ok], b[ok])
    return t, p, int(ok.sum()), float(np.mean(a[ok] - b[ok]))

DEC = {env: decay_table(env) for env in ("nov", "fam")}
for env in ("nov", "fam"):
    d = DEC[env]
    print(f"== {env}: decay rates (1/s)")
    print(d[[c for c in d if not c.endswith("r0.1")]].round(3).to_string())
    for w in ("a60", "b100"):
        for m10 in ("min10", "min10alt"):
            t, p, n, md_ = paired(d[f"{w}_min1"].values, d[f"{w}_{m10}"].values)
            faster = int((d[f"{w}_min1"] > d[f"{w}_{m10}"]).sum())
            print(f"  {env} window {w}, minute 1 vs {m10}: mean difference {md_:+.3f} 1/s, t({n-1}) = {t:.2f}, p = {p:.3g}; minute 1 faster in {faster} of {n}")
    print("  rates above 100 1/s (tau under 10 ms):", int((d[[c for c in d if not c.endswith('r0.1')]] > 100).sum().sum()))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
for k, env in enumerate(("nov", "fam")):
    for start, col in ((0, "k"), (600, "b")):
        curves = []
        for sid in TEST:
            M = MEAS[(sid, env, "pyr")]
            lag, r = S.decorrelation_curve(M["zn"], M["t"], start, 60.0)
            curves.append(r)
        c = np.nanmean(curves, 0)
        ax[k].semilogx(lag, c, col, label=f"minute {'1' if start == 0 else '10'}")
        print(env, start, "mean curve range", np.nanmin(c).round(3), np.nanmax(c).round(3))
    ax[k].set_title({"nov": "novel", "fam": "familiar"}[env]); ax[k].set_xlabel("lag (s)"); ax[k].set_ylabel("mean R")
    ax[k].legend()
plt.tight_layout(); plt.show()

In [ ]:
# the same on the seven sessions whose tracking passes the clock check (position is not used here)
for env in ("nov", "fam"):
    d = DEC[env].loc[CLOCK_OK]
    t, p, n, md_ = paired(d["a60_min1"].values, d["a60_min10"].values)
    print(f"{env}, 7 sessions, reading (a) 600-660: difference {md_:+.3f}, t({n-1}) = {t:.2f}, p = {p:.3g}")

**Result: figs6b-decorrelation-novel is not reproduced; figs6d-decorrelation-familiar is reproduced.** In the
novel run the decay rate of minute 1 is not higher than that of minute 10 under any reading: with 60 s
windows the difference is +0.022 1/s (t(7) = 0.18, p = 0.865) with minute 10 at 600 s and -0.001 (p = 0.987)
at 540 s; with 100 s windows +0.119 (p = 0.25) and +0.096 (p = 0.234). Minute 1 is faster in 3 of 8 sessions
under reading (a). In the familiar run the difference is +0.062 1/s, t(7) = 0.78, p = 0.459 under the judged
reading, and p = 0.40 to 0.72 under the others, so no difference is found there, as the paper reports; the
contrast the paper draws between the two environments is not seen, as the novel run shows no difference
either. No fit gave a time constant under 10 ms.

## fig7ab-umap-embedding

> A 2D UMAP embedding of 100 ms population vectors maps the spatial layout of the environment, and moments just after transition lie at the periphery of the representational space.

UMAP (umap-learn defaults) on the z-scored pyramidal vectors of the first 1000 s of each novel run, every
fifth bin (2000 vectors; a reduction for time). Position mapping: 10-nearest-neighbour regression of the
linearized position from the embedding, 5 contiguous folds, R^2 against the same with the position
permuted in 10 s blocks. Periphery: mean distance from the embedding's median of first-minute bins
against bins from 60 to 1000 s. Reproduced when both hold in most sessions. Run under the claim's seed and
four more.

In [ ]:
import umap
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import KFold, cross_val_predict

def umap_measures(sid, rs, rng):
    M = MEAS[(sid, "nov", "pyr")]
    sel = np.where(M["t"] < 1000)[0][::5]
    E = umap.UMAP(random_state=rs).fit_transform(M["z"][sel])
    t = M["t"][sel]; lin = M["beh"]["lin"][sel]
    ok = np.isfinite(lin)
    knn = KNeighborsRegressor(10)
    pred = cross_val_predict(knn, E[ok], lin[ok], cv=KFold(5))
    r2 = 1 - np.mean((pred - lin[ok]) ** 2) / np.var(lin[ok])
    blocks = (t[ok] // 10).astype(int)
    ub = np.unique(blocks); perm = dict(zip(ub, rng.permutation(ub)))
    shuf = np.concatenate([lin[ok][blocks == perm[b]] for b in ub])[: ok.sum()]
    # permuted blocks can differ in length; pad by wrapping
    if len(shuf) < ok.sum():
        shuf = np.resize(shuf, ok.sum())
    pred_s = cross_val_predict(knn, E[ok], shuf, cv=KFold(5))
    r2_s = 1 - np.mean((pred_s - shuf) ** 2) / np.var(shuf)
    d = np.linalg.norm(E - np.median(E, 0), axis=1)
    return dict(session=sid, r2=r2, r2_shuffled=r2_s, dist_min1=d[t < 60].mean(), dist_later=d[t >= 60].mean(),
                n=len(sel), located=int(ok.sum())), E, t, lin

t0 = time.time()
UM = {}
for k in range(5):
    sd = seed("fig7ab-umap-embedding") + 1000 * k
    rng = np.random.default_rng(sd)
    rows = []
    for sid in TEST:
        res, E, t, lin = umap_measures(sid, sd % (2**31), rng)
        rows.append(res)
        if k == 0 and sid == "e16-3m2-211212":
            EX = (E, t, lin)
        if k == 0 and "EX" not in globals():
            EX = (E, t, lin)
    UM[sd] = pd.DataFrame(rows).set_index("session")
    UM[sd]["maps_position"] = UM[sd]["r2"] > UM[sd]["r2_shuffled"]
    UM[sd]["early_peripheral"] = UM[sd]["dist_min1"] > UM[sd]["dist_later"]
print(f"UMAP runs in {time.time() - t0:.0f} s")
for sd, d in UM.items():
    print(f"seed {sd}: position mapped in {d['maps_position'].sum()}/8, first minute peripheral in {d['early_peripheral'].sum()}/8 "
          f"(clock-checked 7: {d.loc[CLOCK_OK, 'maps_position'].sum()}/7, {d.loc[CLOCK_OK, 'early_peripheral'].sum()}/7)")
print(UM[seed("fig7ab-umap-embedding")].round(3).to_string())

In [ ]:
E, t, lin = EX
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
a = ax[0].scatter(E[:, 0], E[:, 1], c=lin, s=3, cmap="viridis"); plt.colorbar(a, ax=ax[0], label="linearized position (cm)")
b = ax[1].scatter(E[:, 0], E[:, 1], c=t, s=3, cmap="magma"); plt.colorbar(b, ax=ax[1], label="time from transfer (s)")
ax[0].set_title("e16-3m2-211212, novel linear track"); ax[1].set_title("same, by time")
plt.tight_layout(); plt.show()
print("position", np.nanmin(lin).round(1), np.nanmax(lin).round(1), "time", t.min().round(1), t.max().round(1))

**Result: partially reproduced.** Under the claim's seed the embedding predicts the linearized position
better than the block-shuffled position in 7 of 8 sessions (R^2 0.185 to 0.717 against -0.194 to -0.108),
and in 7 or 8 of 8 under all five seeds; the exception is e16-3m2-211211, whose tracking is misfiled (data
check). First-minute bins are farther from the embedding's median than later bins in 1 of 8 sessions (0 or 1
of 8 under all five seeds): by this measure, moments just after transfer are not at the periphery.

## fig7e-novel-settling, fig7f-familiar-settling, fig7g-familiar-more-similar-early

> In a novel environment, population vectors in the first minute after transfer correlate less with their nearest neighbors at the same location than those 10 minutes into the session.
> In the familiar environment, too, ...
> Initial representations are more correlated with their nearest neighbors in the familiar environment than in the novel one.

Per bin, the mean R of the 3 most correlated vectors in the same 1 cm location bin (at least 1 s of
occupancy), smoothed with a 1 s Gaussian; per session the mean over minute 1 and minute 10. Readings: E5
all bins or bins above 5 cm/s; E6 minute 10 as 600 or 540 s; the session set (8 as deposited, or the 7
whose tracking passes the clock check). Paired t-tests over sessions.

In [ ]:
def nn_table(units="pyr", key="nn_s"):
    rows = []
    for sid in TEST:
        for env in ("nov", "fam"):
            M = MEAS[(sid, env, units)]
            for spd in ("all", "fast"):
                x = M[spd][key]
                rows.append(dict(session=sid, env=env, speed=spd,
                                 min1=S.window_mean(x, M["t"], 0), min10=S.window_mean(x, M["t"], 600),
                                 min10alt=S.window_mean(x, M["t"], 540),
                                 **{f"m{k+1}": S.window_mean(x, M["t"], 60 * k) for k in range(15)}))
    return pd.DataFrame(rows)

NN = nn_table()
print(NN[["session", "env", "speed", "min1", "min10", "min10alt"]].round(3).to_string())
FIG7 = []
for sset_name, sset in (("8 sessions", TEST), ("7 clock-checked", CLOCK_OK)):
    for spd in ("all", "fast"):
        for m10 in ("min10", "min10alt"):
            for env in ("nov", "fam"):
                d = NN[(NN.env == env) & (NN.speed == spd) & NN.session.isin(sset)]
                t, p, n, md_ = paired(d["min1"].values, d[m10].values)
                FIG7.append(dict(claim="fig7e" if env == "nov" else "fig7f", sessions=sset_name, speed=spd, minute10=m10,
                                 diff_min1_minus_min10=md_, t=t, df=n - 1, p=p))
            dn = NN[(NN.env == "nov") & (NN.speed == spd) & NN.session.isin(sset)].set_index("session")
            df_ = NN[(NN.env == "fam") & (NN.speed == spd) & NN.session.isin(sset)].set_index("session")
            if m10 == "min10":
                t, p, n, md_ = paired(df_["min1"].values, dn.loc[df_.index, "min1"].values)
                FIG7.append(dict(claim="fig7g", sessions=sset_name, speed=spd, minute10="-", diff_min1_minus_min10=md_, t=t, df=n - 1, p=p))
FIG7 = pd.DataFrame(FIG7)
print(FIG7.round(4).to_string())

In [ ]:
# variants, not judged: neighbours from other visits only, and all units
for label, units, key in (("other visits only", "pyr", "nnx_s"), ("all units", "all", "nn_s")):
    d = nn_table(units, key)
    for env in ("nov", "fam"):
        x = d[(d.env == env) & (d.speed == "all")]
        t, p, n, md_ = paired(x["min1"].values, x["min10"].values)
        print(f"{label}, {env}: min1 {x['min1'].mean():.3f}, min10 {x['min10'].mean():.3f}, t({n-1}) = {t:.2f}, p = {p:.3g}")
    a = d[(d.env == "fam") & (d.speed == "all")].set_index("session")["min1"]; b = d[(d.env == "nov") & (d.speed == "all")].set_index("session")["min1"]
    t, p, n, md_ = paired(a.values, b.loc[a.index].values)
    print(f"{label}, familiar vs novel minute 1: difference {md_:+.3f}, t({n-1}) = {t:.2f}, p = {p:.3g}")

In [ ]:
# variant, not judged: time 0 at the first tracked frame in place of the epoch start (claims file E2)
rows = []
for sid in TEST:
    for env in ("nov", "fam"):
        M = MEAS[(sid, env, "pyr")]; r = SESS[sid]["runs"][0 if env == "fam" else 1]
        off = r["first_track"] - r["t0"]
        x = M["all"]["nn_s"]
        rows.append(dict(session=sid, env=env, offset_s=round(off, 1), min1=S.window_mean(x, M["t"], off),
                         min10=S.window_mean(x, M["t"], off + 600)))
ANCH = pd.DataFrame(rows)
print(ANCH.round(3).to_string())
for env in ("nov", "fam"):
    d = ANCH[ANCH.env == env]
    t, p, n, md_ = paired(d["min1"].values, d["min10"].values)
    print(f"{env}, from the first tracked frame: min1 - min10 = {md_:+.3f}, t({n-1}) = {t:.2f}, p = {p:.3g}")
a = ANCH[ANCH.env == "fam"].set_index("session")["min1"]; b = ANCH[ANCH.env == "nov"].set_index("session")["min1"]
t, p, n, md_ = paired(a.values, b.loc[a.index].values)
print(f"familiar vs novel minute 1, from the first tracked frame: {md_:+.3f}, t({n-1}) = {t:.2f}, p = {p:.3g}")

In [ ]:
# variant, not judged: 1 x 1 cm bins of (x, y), the Methods' bins, for the runs whose x, y are in cm
rows = []
for sid in TEST:
    for env in ("nov", "fam"):
        M = MEAS[(sid, env, "pyr")]; r = SESS[sid]["runs"][0 if env == "fam" else 1]
        loc2 = S.location_2d(SESS[sid], r, M["t"])
        if loc2 is None:
            continue
        nn, _ = S.nn_conditioned(M["zn"], loc2, np.isfinite(loc2), M["t"])
        x = S.nan_smooth(nn)
        rows.append(dict(session=sid, env=env, min1=S.window_mean(x, M["t"], 0), min10=S.window_mean(x, M["t"], 600)))
XY = pd.DataFrame(rows)
print(XY.round(3).to_string())
for env in ("nov", "fam"):
    d = XY[XY.env == env]
    t, p, n, md_ = paired(d["min1"].values, d["min10"].values)
    print(f"{env}, (x, y) bins: min1 - min10 = {md_:+.3f}, t({n-1}) = {t:.2f}, p = {p:.3g}")
both = XY.pivot(index="session", columns="env", values="min1").dropna()
t, p, n, md_ = paired(both["fam"].values, both["nov"].values)
print(f"familiar vs novel minute 1, (x, y) bins: {md_:+.3f}, t({n-1}) = {t:.2f}, p = {p:.3g}")

In [ ]:
# per-minute session means (the 'several minutes' and 'more rapidly' parts, shown, not judged)
mins = [f"m{k+1}" for k in range(15)]
for env in ("nov", "fam"):
    d = NN[(NN.env == env) & (NN.speed == "all")]
    print(env, "mean over sessions by minute:", np.round(d[mins].mean().values, 3))
fig, ax = plt.subplots(figsize=(5, 3))
for env, col in (("nov", "r"), ("fam", "k")):
    d = NN[(NN.env == env) & (NN.speed == "all")]
    ax.plot(range(1, 16), d[mins].mean().values, col, marker="o", label={"nov": "novel", "fam": "familiar"}[env])
ax.set_xlabel("minute after transfer"); ax.set_ylabel("NN correlation (R)"); ax.legend(); plt.tight_layout(); plt.show()
print("by session, novel minute 1 and 10:")
print(NN[(NN.speed == "all")].pivot(index="session", columns="env", values=["min1", "min10"]).round(3).to_string())

**Result: fig7e-novel-settling, fig7f-familiar-settling and fig7g-familiar-more-similar-early are reproduced.**
In the novel run the nearest-neighbour correlation of minute 1 is lower than that of minute 10 by 0.104
(t(6) = -4.07, p = 0.0066; e13-26m1-211119 has no tracking in its first 63 s, so 7 sessions enter), and by
0.106 (t(7) = -6.44, p = 0.000355) with time counted from the first tracked frame; every combination of
readings gives p of 0.026 or less. In the familiar run the difference is 0.089 (t(7) = -5.74, p = 0.0007), p of
0.0053 or less under every combination. Familiar minute 1 is higher than novel minute 1 by 0.037 (t(6) = 2.58,
p = 0.042) over all bins and by 0.079 (t(5) = 4.56, p = 0.0061) above 5 cm/s; on the seven clock-checked
sessions over all bins it is 0.036 (p = 0.089). The per-minute means rise from 0.393 to about 0.50 by minute 7
in the novel run and from 0.430 to about 0.52 by minute 3 in the familiar run, consistent with faster
settling in the familiar environment, which is not judged.

## figs7ab-in-field-out-field

> Settling involves an increase over time of firing within each neuron's place field and a decrease of out-of-field firing, in novel and familiar environments.

Place maps on 1 cm linearized bins above 5 cm/s with more than 1 s occupancy; fields where the
unnormalized rate exceeds 5 Hz; per 100 ms bin, the mean deviation of z-scored rates from the map, in
field and out of field; per session the slope over minutes 1 to 10. Readings: deviations at (a) all bins,
(b) bins above 5 cm/s. One-sample t-tests of the slopes over sessions.

In [ ]:
rows = []
for sid in TEST:
    for env in ("nov", "fam"):
        M = MEAS[(sid, env, "pyr")]
        for rd, spd in (("a", False), ("b", True)):
            t, din, dout, nf = S.field_deviations(M, spd)
            s_in, m_in = S.minute_slope(t, din); s_out, m_out = S.minute_slope(t, dout)
            rows.append(dict(session=sid, env=env, reading=rd, units_with_field=nf, slope_in=s_in, slope_out=s_out,
                             in_min1=m_in[0], in_min10=m_in[-1], out_min1=m_out[0], out_min10=m_out[-1]))
FLD = pd.DataFrame(rows)
print(FLD.round(4).to_string())
for sset_name, sset in (("8 sessions", TEST), ("7 clock-checked", CLOCK_OK)):
    for rd in ("a", "b"):
        for env in ("nov", "fam"):
            d = FLD[(FLD.env == env) & (FLD.reading == rd) & FLD.session.isin(sset)]
            ti, pi = stats.ttest_1samp(d.slope_in, 0, nan_policy="omit"); to, po = stats.ttest_1samp(d.slope_out, 0, nan_policy="omit")
            print(f"{sset_name}, reading {rd}, {env}: in-field slope {d.slope_in.mean():+.4f} (t = {ti:.2f}, p = {pi:.3g}), "
                  f"out-of-field slope {d.slope_out.mean():+.4f} (t = {to:.2f}, p = {po:.3g})")

**Result: partially reproduced.** In-field deviations rise over the first 10 minutes in both environments
(reading a: novel +0.0231 per minute, t = 2.98, p = 0.0207; familiar +0.0212, p = 0.017; reading b: p =
5.68e-05 and 0.000378). Out-of-field deviations fall in the familiar run (reading a -0.0048 per minute, p =
0.00107; reading b p = 0.00402) but not in the novel run (+0.0001, p = 0.966; reading b +0.0017, p = 0.387).
The seven clock-checked sessions give the same pattern.

## figs7cd-movement-not-time-locked

> For the most similar pairs of moments, the absolute differences in velocity and acceleration show no systematic relationship with time after transfer.

Per session, the Spearman correlation of |dvel|NN and |dacc|NN with time over the first 600 s (and the
first 1000 s as a variant); t-test of the mean over sessions against 0. Readings: 3 neighbours or the
single nearest; E8 Kalman or finite differences; E5 all bins or above 5 cm/s. Judged under the first
reading of each (3 neighbours, Kalman, all bins), the others shown.

In [ ]:
rows = []
for sid in TEST:
    for env in ("nov", "fam"):
        M = MEAS[(sid, env, "pyr")]
        for spd in ("all", "fast"):
            for nnk in ("3", "1"):
                for e in ("k", "d"):
                    for meas in ("dvel", "dacc"):
                        x = M[spd][f"{meas}{nnk}_{e}"]
                        for win in (600, 1000):
                            m = (M["t"] < win) & np.isfinite(x)
                            rho = stats.spearmanr(M["t"][m], x[m])[0] if m.sum() > 10 else np.nan
                            rows.append(dict(session=sid, env=env, speed=spd, nn=nnk, filt=e, measure=meas, window=win, rho=rho))
MOV = pd.DataFrame(rows)
summ = []
for key, d in MOV.groupby(["env", "measure", "nn", "filt", "speed", "window"]):
    for sset_name, sset in (("8", TEST), ("7", CLOCK_OK)):
        x = d[d.session.isin(sset)].rho.values
        t, p = stats.ttest_1samp(x, 0, nan_policy="omit")
        summ.append(dict(zip(["env", "measure", "nn", "filt", "speed", "window"], key), sessions=sset_name, mean_rho=np.nanmean(x), t=t, p=p))
MOVS = pd.DataFrame(summ)
main = MOVS[(MOVS.nn == "3") & (MOVS.filt == "k") & (MOVS.speed == "all") & (MOVS.window == 600)]
print("main reading (3 neighbours, Kalman, all bins, first 600 s):")
print(main.round(4).to_string())
print("\nevery reading, 8 sessions, first 600 s: number of tests with p < 0.05 out of", len(MOVS[(MOVS.sessions == "8") & (MOVS.window == 600)]),
      ":", int((MOVS[(MOVS.sessions == "8") & (MOVS.window == 600)].p < 0.05).sum()))
print(MOVS[(MOVS.sessions == "8")].round(3).to_string())

**Result: not reproduced.** In the novel run the movement differences of the nearest neighbours fall with
time after transfer: under the judged reading the mean Spearman correlation is -0.110 for |dvel|NN (t(7) =
-5.42, p = 0.0010) and -0.103 for |dacc|NN (t(7) = -5.77, p = 0.0007). In the familiar run there is none (-0.016,
p = 0.537; -0.005, p = 0.842). Over every reading, 14 of the 32 tests on 8 sessions over 600 s find a
relationship, all of them in the novel run.

## figs7ef-time-explains-similarity

> Modeling nearest-neighbor similarity as a function of time from transfer, |dvel|NN and |dacc|NN, only removing time from transition significantly decreases cross-validated prediction, in novel and familiar environments.

The smoothed NN correlation of each bin, modelled on time from transfer, |dvel|NN and |dacc|NN; each session
withheld in turn; percent change in CVMSE on dropping each variable, one-sample t-test against 0 over
sessions. Readings: (a) the photometry Methods' form (a fast positive and a slow negative exponential of
time, rates bounded at 0.001 to 0.1 and 0.0001 to 0.001 per s, linear movement terms), bounded least
squares; (b) linear in all three; with the E8 and neighbour readings. Bins are taken every 1 s (every
tenth), as the series is smoothed with a 1 s Gaussian.

In [ ]:
t0 = time.time()
rows = []
for form in ("exp", "linear"):
    for nnk in ("3", "1"):
        for e in ("k", "d"):
            for env in ("nov", "fam"):
                for sset_name, sset in (("8", TEST), ("7", CLOCK_OK)):
                    data = []
                    for sid in sset:
                        M = MEAS[(sid, env, "pyr")]
                        y = M["all"]["nn_s"]; dv = M["all"][f"dvel{nnk}_{e}"]; da = M["all"][f"dacc{nnk}_{e}"]
                        m = np.isfinite(y) & np.isfinite(dv) & np.isfinite(da)
                        i = np.where(m)[0][::10]
                        data.append((M["t"][i], dv[i], da[i], y[i]))
                    pc = S.cv_percent_change(data, form)
                    for drop, v in pc.items():
                        t, p = stats.ttest_1samp(v, 0)
                        rows.append(dict(form=form, nn=nnk, filt=e, env=env, sessions=sset_name, dropped=drop,
                                         mean_pct=v.mean(), t=t, p=p, n=len(v)))
CV = pd.DataFrame(rows)
print(f"model fits in {time.time() - t0:.0f} s")
print(CV.round(4).to_string())
def verdict(d):
    ok = {}
    for env in ("nov", "fam"):
        x = d[d.env == env].set_index("dropped")
        ok[env] = (x.loc["time", "p"] < 0.05 and x.loc["time", "mean_pct"] > 0
                   and x.loc["vel", "p"] >= 0.05 and x.loc["acc", "p"] >= 0.05)
    return ok
for key, d in CV.groupby(["form", "nn", "filt", "sessions"]):
    print(key, verdict(d))

**Result: not reproduced.** Under no reading is time the only variable whose removal significantly worsens
the prediction in both environments. With the photometry model's form, 3 neighbours and the Kalman filter,
dropping |dvel|NN raises the CVMSE by 7.76% in the novel run (t = 3.50, p = 0.0100) and 6.96% in the
familiar run (p = 0.0003); dropping time raises it by 7.31% in the novel run (p = 0.0290) and 1.00% in the
familiar run (p = 0.6086); dropping |dacc|NN does not matter (p = 0.67 and 0.51). The other readings give the
same pattern.

## figs7gh-unconditioned-similarity

> When representational similarity is not conditioned on location, first-minute patterns are still less correlated with the rest of the session than those 10 minutes in, in novel and familiar environments.

Readings: (a) the 3 most correlated vectors anywhere in the run, (b) the mean correlation with all other
vectors of the run; both smoothed with a 1 s Gaussian; minute 10 as 600 or 540 s. Paired t-tests.

In [ ]:
rows = []
for sid in TEST:
    for env in ("nov", "fam"):
        M = MEAS[(sid, env, "pyr")]
        for rd, key in (("a", "uncond_top3_s"), ("b", "uncond_all_s")):
            x = M[key]
            rows.append(dict(session=sid, env=env, reading=rd, min1=S.window_mean(x, M["t"], 0),
                             min10=S.window_mean(x, M["t"], 600), min10alt=S.window_mean(x, M["t"], 540)))
UNC = pd.DataFrame(rows)
print(UNC.round(4).to_string())
for rd in ("a", "b"):
    for m10 in ("min10", "min10alt"):
        for env in ("nov", "fam"):
            d = UNC[(UNC.env == env) & (UNC.reading == rd)]
            t, p, n, md_ = paired(d["min1"].values, d[m10].values)
            print(f"reading {rd}, {m10}, {env}: min1 - min10 = {md_:+.4f}, t({n-1}) = {t:.2f}, p = {p:.3g}")

**Result: reproduced under reading (a).** With the 3 most correlated vectors anywhere in the run, minute 1
is lower than minute 10 in the novel run (-0.0236, t(7) = -3.01, p = 0.0198) and the familiar run (-0.0378,
t(7) = -4.66, p = 0.00231), and also with minute 10 at 540 s (p = 0.00301 and 0.000807). Reading (b), the mean
correlation with all other vectors, is near 0 in every window (-0.0016 to 0.0023), as each unit is z-scored and
the vectors then average to about zero, and finds no difference (p = 0.13 and 0.604).

## Summary

| Claim | Outcome |
|---|---|
| Photometry claims (13) and ne-and-settling-time-course | blocked, data missing |
| ephys-ensemble-size | reproduced (all units) |
| figs6b-decorrelation-novel | not reproduced |
| figs6d-decorrelation-familiar | reproduced |
| fig7ab-umap-embedding | partially reproduced (position yes, periphery no) |
| fig7e-novel-settling | reproduced |
| fig7f-familiar-settling | reproduced |
| fig7g-familiar-more-similar-early | reproduced |
| figs7ab-in-field-out-field | partially reproduced (no out-of-field decrease in novel runs) |
| figs7cd-movement-not-time-locked | not reproduced (novel runs) |
| figs7ef-time-explains-similarity | not reproduced (velocity also matters) |
| figs7gh-unconditioned-similarity | reproduced (reading a) |
| fig8b, fig8d, fig8-no-pre-reactivation | in `reactivation.ipynb` |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the
differences from the paper's method.

How the analysis differs from the paper's: pyramidal cells come from the later version's CellExplorer
labels; locations are 1 cm bins of the linearized position, as the x, y position is not in cm in five novel
runs; the ripples are the deposited ones; e13-26m1-211119's first minute has no tracking; the readings the
paper leaves open are each run and shown.

What was found about the dataset (pinned version 0.230630.2304, and the later 0.261113.1035):

- No cell types in the pinned version (`finding-no-cell-types`); the later version adds them.
- The x, y position of five novel runs is in 0 to 1 units, labelled cm, in both versions (`finding-position-not-cm`).
- In e16-3m2-211211 the linear-track tracking and trials are filed in the open-field epoch; spikes are tuned
  to them only when they are moved 5564.5 s later onto the linear-track epoch; the same in the later version
  (`finding-tracking-misfiled`).
- e13-26m1-211116 has no epochs; three sessions have no tracking of their second maze epoch, and four have
  no tracking of one of their two novel environments.
- The behavior module's description spells "alternation" as separate letters in three files (`finding-behavior-description`).

In [ ]:
print(f"notebook runtime {(time.time() - T_START) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e6:.1f} GB")